# 🚀 Financial RAG System: Hybrid Search over SEC Filings + News (BM25 + FAISS + Reranking)

### 📊 Hybrid Retrieval over SEC Filings + Financial News

![Python](https://img.shields.io/badge/Python-3.10-blue?logo=python)
![Kaggle](https://img.shields.io/badge/Kaggle-Notebook-blue?logo=kaggle)
![RAG](https://img.shields.io/badge/AI-RAG-orange)
![FAISS](https://img.shields.io/badge/Vector%20Search-FAISS-green)
![BM25](https://img.shields.io/badge/Retrieval-BM25-yellow)
![Transformers](https://img.shields.io/badge/Models-Transformers-red)

---

## 🔥 Why This Project?

Financial insights are **fragmented** across:

- 📄 SEC 10-K filings → detailed but long & complex  
- 📰 Financial news → fast but shallow  

👉 This notebook combines both to build a **powerful AI-driven financial assistant** using **Retrieval-Augmented Generation (RAG)**.

---

## 🧠 What You’ll Learn

✔️ How to process **real-world financial datasets**  
✔️ How to build a **hybrid retrieval system (BM25 + embeddings)**  
✔️ How to improve results using **cross-encoder reranking**  
✔️ How to design a **production-ready RAG pipeline**

---

## 🏗️ System Architecture

<pre>
📄 SEC Filings +  📰 News Data
              ↓                 
   Cleaning & Normalization
              ↓
      ✂️ Chunking (SEC)
              ↓
        📦 Unified Corpus
              ↓
   🔎 BM25 + Dense Embeddings
              ↓
      ⚡ Hybrid Retrieval
              ↓
   🎯 Cross-Encoder Reranking
              ↓
     🤖 Context Generation (RAG)
</pre>

## ⚙️ Key Components

### 📄 SEC Filings Processing
- Extract structured sections (Risk Factors, MD&A)
- Optimized chunking with overlap for better retrieval

### 📰 Financial News Integration
- CNBC, Reuters, Guardian
- Cleaned & unified into a single schema

### 🔎 Hybrid Retrieval Engine
- **BM25** → keyword precision  
- **FAISS embeddings** → semantic understanding  
- Combined for best performance

### 🎯 Reranking Layer
- Cross-encoder improves relevance dramatically
- Re-scores query-document pairs

### 🏗️ Production Design
- `CorpusBuilder`
- `HybridRetriever`
- `RAGPipeline`

👉 Modular → reusable → scalable

---

## 🎯 Example Query

> 💬 *"What risk factors did tech companies report in 2021?"*

### 🧠 What happens:
1. Retrieve relevant SEC + news documents  
2. Combine keyword + semantic search  
3. Rerank for accuracy  
4. Build context for LLM  

---

## 💼 Real-World Use Cases

- 📈 Financial Analysts → Risk & trend analysis  
- 💰 Investors → Combine filings + sentiment  
- ⚖️ Compliance Teams → Regulatory insights  
- 🤖 AI Applications → Financial Q&A systems  

---

## 🚀 Highlights

✨ Hybrid search (BM25 + FAISS)  
✨ Cross-encoder reranking  
✨ Real-world financial datasets  
✨ Production-style modular pipeline  
✨ End-to-end RAG workflow  

---

## 🔮 Future Improvements

- 🔗 Integrate LLMs (OpenAI / local models)  
- 📊 Add evaluation metrics (Recall@K, MRR)  
- 🌐 Deploy via FastAPI / Streamlit  
- ⚡ GPU optimization for faster inference  

---

## ⭐ If you found this useful...

- 👍 Upvote the notebook  
- 🔁 Fork and experiment  
- 💬 Share feedback  

---

💡 *This project demonstrates how modern AI systems combine retrieval + reasoning to solve real-world financial problems.*

## 1. Imports and Setup

We use:
- pandas / numpy → data handling
- nltk → sentence splitting
- BM25 → keyword retrieval
- sentence-transformers → embeddings + reranking
- FAISS → fast vector search

In [1]:
import os
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

# Text + NLP
import nltk
from nltk.tokenize import sent_tokenize

# BM25
!pip install rank-bm25 sentence-transformers faiss-cpu --quiet

from rank_bm25 import BM25Okapi

# Embeddings + Cross-Encoder
from sentence_transformers import SentenceTransformer, CrossEncoder

# For vector index (we'll use FAISS)
import faiss

nltk.download('punkt')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.0 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## 2. Configuration

We define:
- Dataset paths
- Model choices
- Chunking parameters

### Why chunking?
SEC filings are very long → we split them into smaller chunks to improve retrieval quality.

In [2]:

NEWS_DATA_DIR = Path("/kaggle/input/datasets/notlucasp/financial-news-headlines")
SEC_DATA_DIR = Path("/kaggle/input/datasets/pranjalverma08/sec-edgar-annual-financial-filings-2021/extracted")

# Model names 
DENSE_MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L3-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# Chunking configuration for SEC filings
MAX_CHARS_PER_CHUNK = 1200   # target chunk size
CHUNK_OVERLAP = 200          # overlap between chunks


## 3. Load Financial News

We combine multiple datasets into a unified schema:
- CNBC
- Guardian
- Reuters


We will:
1. Load each file into a DataFrame  
2. Normalize column names  
3. Add a `source` column  
4. Concatenate into a single `news_df`

This allows consistent retrieval across sources.

In [3]:
def load_news_data(news_dir: Path) -> pd.DataFrame:
    # CNBC
    cnbc = pd.read_csv(news_dir / "cnbc_headlines.csv")
    cnbc = cnbc.rename(columns={
        "Headlines": "headline",
        "Time": "time",
        "Description": "description"
    })
    cnbc["source"] = "cnbc"
    
    # Guardian
    guardian = pd.read_csv(news_dir / "guardian_headlines.csv")
    guardian = guardian.rename(columns={
        "Headlines": "headline",
        "Time": "time"
    })
    guardian["description"] = None
    guardian["source"] = "guardian"
    
    # Reuters
    reuters = pd.read_csv(news_dir / "reuters_headlines.csv")
    reuters = reuters.rename(columns={
        "Headlines": "headline",
        "Time": "time",
        "Description": "description"
    })
    reuters["source"] = "reuters"
    
    # Align columns
    common_cols = ["time", "headline", "description", "source"]
    news_df = pd.concat(
        [cnbc[common_cols], guardian[common_cols], reuters[common_cols]],
        ignore_index=True
    )
    
    return news_df

news_df = load_news_data(NEWS_DATA_DIR)
news_df.head()


,time,headline,description,source
0,"7:51 PM ET Fri, 17 July 2020",Jim Cramer: A better way to invest in the Covi...,"""Mad Money"" host Jim Cramer recommended buying...",cnbc
1,"7:33 PM ET Fri, 17 July 2020",Cramer's lightning round: I would own Teradyne,"""Mad Money"" host Jim Cramer rings the lightnin...",cnbc
2,NaN,NaN,NaN,cnbc
3,"7:25 PM ET Fri, 17 July 2020","Cramer's week ahead: Big week for earnings, ev...","""We'll pay more for the earnings of the non-Co...",cnbc
4,"4:24 PM ET Fri, 17 July 2020",IQ Capital CEO Keith Bliss says tech and healt...,"Keith Bliss, IQ Capital CEO, joins ""Closing Be...",cnbc


### 3.1 Clean News Data

- Remove missing headlines
- Normalize timestamps
- Build a `text` field combining headline + description  
- Remove very short texts
- Filter noisy entries

This improves both BM25 and embeddings.

In [4]:
import re
from datetime import datetime

def parse_news_time(t):
    """
    Robust parser for CNBC/Reuters/Guardian timestamps.
    Normalizes all formats into a single consistent datetime.
    Removes ET timezone, fixes month names, handles inconsistent spacing.
    """
    if not isinstance(t, str):
        return pd.NaT

    # Remove ET timezone
    t = t.replace("ET", "")

    # Normalize whitespace
    t = " ".join(t.split())

    # Fix inconsistent month spelling
    t = t.replace("Sept", "Sep")

    # CNBC formats look like:
    # "7:19 PM Wed, 3 Jan 2018"
    # "10:13 AM Wed, 27 Dec 2017"
    # "11:12 AM Thu, 20 Sep 2018"

    # Regex to extract components
    pattern = r"(\d{1,2}:\d{2})\s*(AM|PM)\s*\w{3},\s*(\d{1,2})\s*(\w{3})\s*(\d{4})"
    m = re.search(pattern, t)

    if m:
        time_str, ampm, day, month, year = m.groups()
        cleaned = f"{day} {month} {year} {time_str} {ampm}"
        try:
            return datetime.strptime(cleaned, "%d %b %Y %I:%M %p")
        except:
            return pd.NaT

    # Fallback to dateutil (rare cases)
    try:
        return pd.to_datetime(t, errors="coerce")
    except:
        return pd.NaT


In [5]:
def clean_news_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    
    # Drop rows without headline
    df = df.dropna(subset=["headline"])
    
    # Parse time using robust parser
    df["time_parsed"] = df["time"].apply(parse_news_time)
    
    # Unified text field
    df["description"] = df["description"].fillna("")
    df["text"] = df["headline"].astype(str) + ". " + df["description"].astype(str)
    
    # Remove very short texts
    df["text_len"] = df["text"].str.len()
    df = df[df["text_len"] > 20]
    
    df = df.reset_index(drop=True)
    return df


news_df = clean_news_data(news_df)
news_df[["source", "time_parsed", "headline", "text"]].head()


,source,time_parsed,headline,text
0,cnbc,2020-07-17 19:51:00,Jim Cramer: A better way to invest in the Covi...,Jim Cramer: A better way to invest in the Covi...
1,cnbc,2020-07-17 19:33:00,Cramer's lightning round: I would own Teradyne,Cramer's lightning round: I would own Teradyne...
2,cnbc,2020-07-17 19:25:00,"Cramer's week ahead: Big week for earnings, ev...","Cramer's week ahead: Big week for earnings, ev..."
3,cnbc,2020-07-17 16:24:00,IQ Capital CEO Keith Bliss says tech and healt...,IQ Capital CEO Keith Bliss says tech and healt...
4,cnbc,2020-07-16 19:36:00,Wall Street delivered the 'kind of pullback I'...,Wall Street delivered the 'kind of pullback I'...


## 4. Load SEC 10-K filings (JSON)

The SEC dataset provides processed JSON files under `JSONExtracted`.  
Each JSON file typically corresponds to one filing and contains:
- Company metadata (CIK, name, form type, date)
- An `items` dictionary with sections like:
  - `1` – Business
  - `1A` – Risk Factors
  - `7` – Management Discussion & Analysis
  - `8` – Financial Statements

We will:
1. Iterate over JSON files  
2. Extract each item section as a separate record  
3. Build a DataFrame `sec_sections_df` with one row per section


In [6]:
def load_sec_sections(sec_dir: Path, max_files: int = None) -> pd.DataFrame:
    records = []
    
    # Find all JSON files
    files = list(sec_dir.glob("*.json"))
    print("Found JSON files:", len(files))
    
    if max_files is not None:
        files = files[:max_files]
    
    for path in files:
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print("Skipping:", path, "Error:", e)
            continue
        
        # Basic metadata
        cik = data.get("cik")
        company_name = data.get("company")
        form_type = data.get("filing_type")
        filing_date = data.get("filing_date")
        period_of_report = data.get("period_of_report")
        sic = data.get("sic")
        
        # Extract all keys that start with "item_"
        for key, value in data.items():
            if key.lower().startswith("item_") and isinstance(value, str):
                records.append({
                    "cik": cik,
                    "company_name": company_name,
                    "form_type": form_type,
                    "filing_date": filing_date,
                    "period_of_report": period_of_report,
                    "sic": sic,
                    "item": key,          # e.g., item_1, item_1a, item_7
                    "raw_text": value
                })
    
    return pd.DataFrame(records)



sec_sections_df = load_sec_sections(SEC_DATA_DIR, max_files=200)  # limit for Kaggle runtime
sec_sections_df.head()


Found JSON files: 191


,cik,company_name,form_type,filing_date,period_of_report,sic,item,raw_text
0,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,2021-09-30,7372,item_1,Item 1. Business.\nOverview\nBridgeline Digita...
1,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,2021-09-30,7372,item_1A,Item 1A. Risk Factors\nThis report contains fo...
2,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,2021-09-30,7372,item_1B,Item 1B. Unresolved Staff Comments\nNone
3,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,2021-09-30,7372,item_2,Item 2. Properties.\nThe following table lists...
4,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,2021-09-30,7372,item_3,"Item 3. Legal Proceedings.\nFrom time to time,..."


In [7]:
sec_sections_df = load_sec_sections(SEC_DATA_DIR, max_files=50)
sec_sections_df.head()
sec_sections_df["item"].value_counts()

Found JSON files: 191


item
item_1     50
item_1A    50
item_1B    50
item_2     50
item_3     50
item_4     50
item_5     50
item_6     50
item_7     50
item_7A    50
item_8     50
item_9     50
item_9A    50
item_9B    50
item_10    50
item_11    50
item_12    50
item_13    50
item_14    50
item_15    50
Name: count, dtype: int64

## 5. Chunk SEC Filings

### Why chunking?
- Models have context limits
- Retrieval works better on smaller chunks

### Strategy:
- Sentence-based chunking
- Max size: 1200 chars
- Overlap: 200 chars

Overlap helps preserve context across chunks.

In [8]:
def chunk_text(text: str,
               max_chars: int = 1200,
               overlap: int = 200) -> list:
    sentences = sent_tokenize(text)
    chunks = []
    current = ""

    for sent in sentences:
        if len(current) + len(sent) + 1 <= max_chars:
            current += " " + sent if current else sent
        else:
            chunks.append(current.strip())
            if overlap > 0:
                tail = current[-overlap:]
                current = tail + " " + sent
            else:
                current = sent

    if current:
        chunks.append(current.strip())

    return chunks


def build_sec_chunks(sec_df: pd.DataFrame,
                     max_chars: int = 1200,
                     overlap: int = 200) -> pd.DataFrame:
    records = []
    for _, row in sec_df.iterrows():
        text = row["raw_text"]
        if not isinstance(text, str) or len(text) < 50:
            continue

        chunks = chunk_text(text, max_chars=max_chars, overlap=overlap)
        for i, chunk in enumerate(chunks):
            records.append({
                "cik": row["cik"],
                "company_name": row["company_name"],
                "form_type": row["form_type"],
                "filing_date": row["filing_date"],
                "item": row["item"],
                "chunk_id": f"{row['cik']}_{row['item']}_{i}",
                "text": chunk
            })

    return pd.DataFrame(records)


sec_chunks_df = build_sec_chunks(sec_sections_df)
sec_chunks_df.head()


,cik,company_name,form_type,filing_date,item,chunk_id,text
0,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_0,Item 1. Business. Overview\nBridgeline Digital...
1,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_1,"combining content with business data, processe..."
2,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_2,er's site search and browse experience. Hawk S...
3,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_3,": Boston, Massachusetts; Woodbury, New York; C..."
4,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_4,nbound platform empowers companies and develop...


## 6. Build Unified Corpus

We merge:

- SEC chunks (long-form, structured)  
- News articles (short-form, time-sensitive)

Into a single searchable dataset.

In [9]:
def build_corpus(sec_chunks: pd.DataFrame,
                 news: pd.DataFrame) -> pd.DataFrame:
    sec_part = sec_chunks.copy()
    sec_part["doc_id"] = "sec_" + sec_part["chunk_id"].astype(str)
    sec_part["source_type"] = "sec"

    news_part = news.copy()
    news_part["doc_id"] = "news_" + news_part.index.astype(str)
    news_part["source_type"] = "news"

    news_part["company_name"] = None
    news_part["form_type"] = None
    news_part["item"] = None
    news_part["chunk_id"] = None
    news_part["filing_date"] = None

    corpus_df = pd.concat([sec_part, news_part], ignore_index=True)
    return corpus_df


corpus_df = build_corpus(sec_chunks_df, news_df)
corpus_df.head()


,cik,company_name,form_type,filing_date,item,chunk_id,text,doc_id,source_type,time,headline,description,source,time_parsed,text_len
0,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_0,Item 1. Business. Overview\nBridgeline Digital...,sec_1378590_item_1_0,sec,NaN,NaN,NaN,NaN,NaT,NaN
1,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_1,"combining content with business data, processe...",sec_1378590_item_1_1,sec,NaN,NaN,NaN,NaN,NaT,NaN
2,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_2,er's site search and browse experience. Hawk S...,sec_1378590_item_1_2,sec,NaN,NaN,NaN,NaN,NaT,NaN
3,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_3,": Boston, Massachusetts; Woodbury, New York; C...",sec_1378590_item_1_3,sec,NaN,NaN,NaN,NaN,NaT,NaN
4,1378590,"Bridgeline Digital, Inc.",10-K,2021-12-20,item_1,1378590_item_1_4,nbound platform empowers companies and develop...,sec_1378590_item_1_4,sec,NaN,NaN,NaN,NaN,NaT,NaN


## 7. BM25 index (sparse retrieval)

We build a BM25 index over tokenized text for classic lexical matching.

BM25 works well for:
- Exact keyword matches
- Financial terminology


In [10]:
def simple_tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

corpus_tokens = [simple_tokenize(t) for t in corpus_df["text"].tolist()]
bm25 = BM25Okapi(corpus_tokens)

def bm25_search(query, top_k=20):
    q = simple_tokenize(query)
    scores = bm25.get_scores(q)
    idx = np.argsort(scores)[::-1][:top_k]
    return idx, scores[idx]


## 8. Dense Retrieval (Embeddings + FAISS)

We encode all documents using a SentenceTransformer model and store them in a FAISS
inner-product index. For Kaggle stability, we run embeddings on CPU.

Embeddings capture:
- Semantic meaning
- Context similarity

FAISS enables fast vector search.

In [11]:
dense_model = SentenceTransformer(DENSE_MODEL_NAME)
device = "cpu"
print("Using device:", device)

from tqdm import tqdm

BATCH = 64
embeddings = []

texts = corpus_df["text"].tolist()

for i in tqdm(range(0, len(texts), BATCH)):
    batch = texts[i:i+BATCH]
    emb = dense_model.encode(
        batch,
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=device
    )
    embeddings.append(emb)

embeddings = np.vstack(embeddings).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

def dense_search(query, top_k=20):
    q_emb = dense_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
        device=device
    )
    scores, idx = index.search(q_emb, top_k)
    return idx[0], scores[0]


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Using device: cpu


100%|██████████| 1144/1144 [08:37<00:00,  2.21it/s]


## 9. Hybrid search (BM25 + dense)

We combine:
- BM25 (keyword strength)
- Dense (semantic strength) cores
via min-max normalization 

Final score:
weighted combination of both

In [12]:
def minmax(x):
    x = np.array(x)
    if x.max() == x.min():
        return np.zeros_like(x)
    return (x - x.min()) / (x.max() - x.min())


def hybrid_search(query, top_k=20, alpha=0.5):
    b_idx, b_scores = bm25_search(query, top_k)
    d_idx, d_scores = dense_search(query, top_k)

    b_norm = minmax(b_scores)
    d_norm = minmax(d_scores)

    scores = {}
    for i, s in zip(b_idx, b_norm):
        scores[int(i)] = scores.get(int(i), 0) + alpha * s
    for i, s in zip(d_idx, d_norm):
        scores[int(i)] = scores.get(int(i), 0) + (1 - alpha) * s

    sorted_ids = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_ids = [i for i, _ in sorted_ids[:top_k]]
    return np.array(top_ids)


## 10. Cross-Encoder Reranking

Why rerank?

We use a cross-encoder to rerank the top hybrid candidates by directly scoring
(query, document) pairs. For Kaggle stability, we run it on CPU.

Initial retrieval = rough candidates  
Cross-encoder = precise scoring

This significantly improves answer quality.

In [13]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")

def rerank(query, candidate_ids, top_k=10):
    texts = corpus_df.loc[candidate_ids, "text"].tolist()
    pairs = [(query, t) for t in texts]
    scores = cross_encoder.predict(pairs)   # CPU-safe
    order = np.argsort(scores)[::-1]
    return candidate_ids[order][:top_k]



config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

## 11. End-to-end retrieval

We combine hybrid search + reranking into a single retrieval function that
returns the top documents for a query.


In [14]:
def retrieve_documents(query, top_k=10):
    hybrid_ids = hybrid_search(query, top_k=50)
    reranked = rerank(query, hybrid_ids, top_k=top_k)
    return corpus_df.loc[reranked].reset_index(drop=True)


## 12. RAG answer stub

We build a context string from retrieved documents. In a full application,
this context would be sent to an LLM to generate a final answer.

In [15]:
def build_context(results, max_chars=3000):
    context = ""
    for t in results["text"]:
        if len(context) + len(t) > max_chars:
            break
        context += t + "\n\n---\n\n"
    return context


def rag_answer_stub(query, top_k=5):
    docs = retrieve_documents(query, top_k)
    context = build_context(docs)

    answer = f"""
Query: {query}

Context (truncated):
{context[:1200]}

In production, this context is sent to an LLM for answer generation.
"""
    return answer, docs


## 13. Demo query

We test the full pipeline on a realistic financial question.


In [16]:
query = "What risk factors did tech companies report in 2021?"
answer, docs = rag_answer_stub(query)
print(answer)
docs.head()


Query: What risk factors did tech companies report in 2021?

Context (truncated):
r our risk factors for any reason. The following section generally discusses fiscal 2021 and 2020 items and year-to-year comparisons between fiscal 2021 and 2020, as well as certain fiscal 2019 items. Discussions of fiscal 2019 items and year-to-year comparisons between fiscal 2020 and 2019 that are not included in this Form 10-K can be found in “Management’s Discussion and Analysis of Financial Condition and Results of Operations” in Part II, Item 7 of our Annual Report on Form 10-K for the fiscal year ended January 31, 2020. Overview
We are a global leader in customer relationship management ("CRM") technology that brings companies and customers together. With our Customer 360 platform we deliver a single source of truth, connecting customer data across systems, apps and devices to help companies sell, service, market and conduct commerce, from anywhere. Since our founding in 1999, we have pioneered in

,cik,company_name,form_type,filing_date,item,chunk_id,text,doc_id,source_type,time,headline,description,source,time_parsed,text_len
0,1108524,"SALESFORCE.COM, INC.",10-K,2021-03-17,item_7,1108524_item_7_1,r our risk factors for any reason. The followi...,sec_1108524_item_7_1,sec,NaN,NaN,NaN,NaN,NaT,NaN
1,1029744,SONIC FOUNDRY INC,10-K,2021-12-09,item_1A,1029744_item_1A_2,"ber 30, 2021\nRisk Factor Summary Disclosure\n...",sec_1029744_item_1A_2,sec,NaN,NaN,NaN,NaN,NaT,NaN
2,814547,FAIR ISAAC CORP,10-K,2021-11-10,item_7,814547_item_7_1,"with Item 8, Financial Statements and Suppleme...",sec_814547_item_7_1,sec,NaN,NaN,NaN,NaN,NaT,NaN
3,1029744,SONIC FOUNDRY INC,10-K,2021-12-09,item_1A,1029744_item_1A_1,equired to make pursuant to Regulation S-K. Th...,sec_1029744_item_1A_1,sec,NaN,NaN,NaN,NaN,NaT,NaN
4,1348036,"AVALARA, INC.",10-K,2021-02-25,item_1A,1348036_item_1A_0,Item 1A. Risk Factors. You should carefully co...,sec_1348036_item_1A_0,sec,NaN,NaN,NaN,NaN,NaT,NaN


## 14. Production-style modular design

In a real application (Streamlit, FastAPI, etc.), we would wrap the logic into
classes such as:

- `CorpusBuilder` — loads and preprocesses datasets  
- `HybridRetriever` — BM25 + dense + reranking  
- `RAGPipeline` — high-level interface for retrieval + answer generation  

This modular design makes it easy to reuse the pipeline in different apps.
Below is a minimal skeleton that can be extended.


In [17]:
class CorpusBuilder:
    def __init__(self, news_dir, sec_dir,
                 max_sec_files=50,
                 max_chars=1200,
                 overlap=200):
        self.news_dir = news_dir
        self.sec_dir = sec_dir
        self.max_sec_files = max_sec_files
        self.max_chars = max_chars
        self.overlap = overlap

    def build(self):
        news_df = load_news_data(self.news_dir)
        news_df = clean_news_data(news_df)
        sec_df = load_sec_sections(self.sec_dir, max_files=self.max_sec_files)
        sec_chunks = build_sec_chunks(sec_df, self.max_chars, self.overlap)
        corpus_df = build_corpus(sec_chunks, news_df)
        return corpus_df


class HybridRetriever:
    def __init__(self, corpus_df, dense_model, cross_encoder):
        self.corpus_df = corpus_df.reset_index(drop=True)
        self.dense_model = dense_model
        self.cross_encoder = cross_encoder

        self.tokens = [simple_tokenize(t) for t in self.corpus_df["text"]]
        self.bm25 = BM25Okapi(self.tokens)

        self.embeddings = self._encode_corpus()
        dim = self.embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(self.embeddings)

    def _encode_corpus(self, batch=64):
        all_emb = []
        texts = self.corpus_df["text"].tolist()
        for i in tqdm(range(0, len(texts), batch)):
            batch_texts = texts[i:i+batch]
            emb = self.dense_model.encode(
                batch_texts,
                convert_to_numpy=True,
                normalize_embeddings=True,
                device="cpu"
            )
            all_emb.append(emb)
        return np.vstack(all_emb).astype("float32")

    def bm25_search(self, query, top_k=20):
        q = simple_tokenize(query)
        scores = self.bm25.get_scores(q)
        idx = np.argsort(scores)[::-1][:top_k]
        return idx, scores[idx]

    def dense_search(self, query, top_k=20):
        q_emb = self.dense_model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
            device="cpu"
        )
        scores, idx = self.index.search(q_emb, top_k)
        return idx[0], scores[0]

    def hybrid_search(self, query, top_k=20, alpha=0.5):
        b_idx, b_scores = self.bm25_search(query, top_k)
        d_idx, d_scores = self.dense_search(query, top_k)

        b_norm = minmax(b_scores)
        d_norm = minmax(d_scores)

        scores = {}
        for i, s in zip(b_idx, b_norm):
            scores[int(i)] = scores.get(int(i), 0) + alpha * s
        for i, s in zip(d_idx, d_norm):
            scores[int(i)] = scores.get(int(i), 0) + (1 - alpha) * s

        sorted_ids = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        top_ids = [i for i, _ in sorted_ids[:top_k]]
        return np.array(top_ids)

    def rerank(self, query, candidate_ids, top_k=10):
        texts = self.corpus_df.loc[candidate_ids, "text"].tolist()
        pairs = [(query, t) for t in texts]
        scores = self.cross_encoder.predict(pairs)
        order = np.argsort(scores)[::-1]
        return candidate_ids[order][:top_k]

    def retrieve(self, query, top_k=10):
        hybrid_ids = self.hybrid_search(query, top_k=50)
        reranked = self.rerank(query, hybrid_ids, top_k)
        return self.corpus_df.loc[reranked].reset_index(drop=True)


class RAGPipeline:
    def __init__(self, news_dir, sec_dir):
        builder = CorpusBuilder(news_dir, sec_dir)
        self.corpus_df = builder.build()
        self.dense_model = SentenceTransformer(DENSE_MODEL_NAME)
        self.cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")
        self.retriever = HybridRetriever(self.corpus_df,
                                         self.dense_model,
                                         self.cross_encoder)

    def answer(self, query, top_k=5):
        docs = self.retriever.retrieve(query, top_k)
        context = build_context(docs)
        return {
            "query": query,
            "context": context,
            "documents": docs
        }


## Conclusion

We built a production-style Financial RAG system combining:
- SEC filings (structured financial data)
- News headlines (real-time signals)

Data → Cleaning → Chunking → Embeddings → Hybrid Search → Rerank → Answer

Key strengths:
- Hybrid retrieval (BM25 + embeddings)
- Cross-encoder reranking
- Modular production design
  

**Future improvements**:
- LLM integration for final answer generation
- Evaluation metrics (Recall@K)
- Deployment via FastAPI or Streamlit